In [ ]:
%run "./00_CC_Mapping_Setup.ipynb"

In [ ]:
%sql
/* ===================================================================================
   METRIC: EBA07 - Marketing (Canada, Hong Kong, Barbados Only)
   
   LOGIC SUMMARY:
   1. FILTERED MASTER LIST: Filters abac_2025.fy25_list_of_unit strictly for Canada, 
      Hong Kong, and Barbados. Pulls ID, Name, Jurisdiction, and Segment.
   2. FINANCE DATA ONLY: Unions the 6 Finance files and filters for Marketing codes.
   3. MAPPING AGGREGATION: Joins the Cost Center Mapping Bootstrap to the transactions 
      to determine which AUs had marketing spend.
   4. FINAL OUTPUT: LEFT JOINS the mapping results onto the Filtered Master List.
      - Outputs 'Yes', 'No', or '' (Blank) if the AU mapping is missing entirely.
=================================================================================== */

WITH Filtered_Master_List AS (
    -- 1. Grab the Master AUs, add the segment, and strictly filter jurisdictions
    SELECT DISTINCT 
        TRIM(CAST(business_ID AS STRING)) AS BusinessID,
        TRIM(business_operational_unit_name) AS AU_Name,
        TRIM(business_segment) AS Business_Segment,
        TRIM(jurisdiction) AS Jurisdiction
    FROM abac_2025.fy25_list_of_unit
    WHERE business_ID IS NOT NULL
      AND TRIM(jurisdiction) IN ('Canada', 'Hong Kong', 'Barbados')
),

Combined_Finance_Raw AS (
    -- 2. Union all 6 Finance files into one master dataset
    SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_1
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_2
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_3
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_4
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_5
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_6
),

Marketing_Transactions AS (
    -- 3. Filter STRICTLY for marketing category codes
    SELECT 
        TRY_CAST(TRIM(CostCenter) AS INT) AS Cleaned_CC
    FROM Combined_Finance_Raw
    WHERE TRIM(CatCode) IN (
        '194', '370', '372', '377', '408', '628', 
        '629', '653', '830', '875', '877'
    )
),

AU_Marketing_Flags AS (
    -- 4. Map transactions to AUs and flag if they have marketing spend
    SELECT 
        TRIM(CAST(m.AU_ID AS STRING)) AS BusinessID,
        MAX(CASE WHEN mt.Cleaned_CC IS NOT NULL THEN 1 ELSE 0 END) AS Has_Marketing
    FROM vw_cost_center_mapping_bootstrap m
    LEFT JOIN Marketing_Transactions mt
        ON CAST(m.Cost_Center_ID AS INT) = mt.Cleaned_CC
    GROUP BY m.AU_ID
)

-- 5. Build final report using the Filtered Master List as the anchor
SELECT 
    fml.BusinessID,                          
    fml.AU_Name, 
    fml.Business_Segment,
    'EBA07' AS QuestionID,               
    
    CASE 
        WHEN amf.Has_Marketing = 1 THEN 'Yes' 
        -- It exists in the mapping table but has NO marketing spend
        WHEN amf.BusinessID IS NOT NULL THEN 'No' 
        -- It ONLY exists in the Master List (missing mapping/data entirely)
        ELSE '' 
    END AS Response,

    -- Since we anchored to the Master List, these are guaranteed to be 'Yes'
    'Yes' AS In_ABAC_AU_List
    
FROM Filtered_Master_List fml

-- LEFT JOIN guarantees we ONLY see the Canada, HK, and Barbados AUs
LEFT JOIN AU_Marketing_Flags amf
    ON fml.BusinessID = amf.BusinessID

ORDER BY fml.BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: EBA07 - Marketing Drill-Down
   
   PURPOSE: 
   Provides a row-by-row view of every Cost Center transaction that triggered a 'Yes' 
   for EBA07. Verifies that the matched Category Code belongs to the marketing list.
=================================================================================== */

WITH Combined_Finance_Raw AS (
    SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_1
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_2
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_3
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_4
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_5
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_6
),

Marketing_Transactions AS (
    SELECT 
        TRY_CAST(TRIM(CostCenter) AS INT) AS Cleaned_CC,
        TRIM(CatCode) AS CatCode,
        CostCenter AS Raw_Cost_Center_String
    FROM Combined_Finance_Raw
    WHERE TRIM(CatCode) IN (
        '194', '370', '372', '377', '408', '628', 
        '629', '653', '830', '875', '877'
    )
)

SELECT DISTINCT
    m.AU_ID AS BusinessID,
    m.AU_Name,
    m.Cost_Center_ID AS Mapped_Cost_Center,
    mt.Raw_Cost_Center_String,
    mt.CatCode AS Extracted_Category_Code,
    CASE 
        WHEN mt.CatCode = '370' THEN 'Charitable Sponsorships'
        WHEN mt.CatCode = '372' THEN 'C&PA Sponsorship-Marketing only'
        WHEN mt.CatCode = '628' THEN 'Sponsorship Fees'
        ELSE 'Other Marketing Code'
    END AS Category_Description
    
FROM vw_cost_center_mapping_bootstrap m

-- INNER JOIN to only return AUs that successfully mapped to a marketing transaction
INNER JOIN Marketing_Transactions mt
    ON CAST(m.Cost_Center_ID AS INT) = mt.Cleaned_CC
    
ORDER BY BusinessID;

In [ ]:
%sql
/* ===================================================================================
   QA DIFF LOGIC: Missing AUs for Canada, Barbados, Hong Kong (EBA07)
   
   PURPOSE: 
   Identifies Business IDs that exist in abac_2025.fy25_list_of_unit for specific 
   jurisdictions but have NO matching marketing transactions in the EBA07 logic.
=================================================================================== */

WITH Filtered_Master_List AS (
    -- 1. Grab the Master AUs and filter jurisdiction for the target countries
    SELECT 
        TRIM(CAST(business_ID AS STRING)) AS BusinessID,
        TRIM(business_operational_unit_name) AS AU_Name,
        TRIM(jurisdiction) AS jurisdiction
    FROM abac_2025.fy25_list_of_unit
    WHERE business_ID IS NOT NULL
      AND TRIM(jurisdiction) IN ('Canada', 'Hong Kong', 'Barbados')
),

Combined_Finance_Raw AS (
    -- 2. Pull the 6 Finance files based on your EBA07 logic
    SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_1
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_2
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_3
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_4
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_5
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_6
),

Marketing_Transactions AS (
    -- 3. Filter for Marketing Category Codes
    SELECT 
        TRY_CAST(TRIM(CostCenter) AS INT) AS Cleaned_CC
    FROM Combined_Finance_Raw
    WHERE TRIM(CatCode) IN (
        '194', '370', '372', '377', '408', '628', 
        '629', '653', '830', '875', '877'
    )
),

EBA07_Successful_AUs AS (
    -- 4. Get the distinct Business IDs that successfully matched to a marketing transaction
    SELECT DISTINCT 
        TRIM(CAST(m.AU_ID AS STRING)) AS BusinessID
    FROM vw_cost_center_mapping_bootstrap m
    INNER JOIN Marketing_Transactions mt
        ON CAST(m.Cost_Center_ID AS INT) = mt.Cleaned_CC
)

-- 5. Find the Difference (What is in the Filtered Master List but NOT in EBA07 Successful AUs)
SELECT 
    fml.BusinessID AS Missing_BusinessID,
    fml.AU_Name,
    fml.jurisdiction,
    'No EBA07 Marketing Transactions Mapped' AS Diff_Reason
    
FROM Filtered_Master_List fml

-- LEFT JOIN targeting NULLs isolates the rows that are missing
LEFT JOIN EBA07_Successful_AUs eba 
    ON fml.BusinessID = eba.BusinessID
WHERE eba.BusinessID IS NULL

ORDER BY fml.BusinessID;